<a href="https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment01/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
# --- Setup ---
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv")
print("Ready.")


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Ready.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

####**Finding A — “Refreshing mature pages produces large impression lifts.”**

**Label / outcome:** post-refresh impression change (often median lift by age × competition segment).

**Methodology question:** How were “refreshed” vs “not refreshed” pages assigned? If editors chose which pages to refresh, part of the lift is selection (they picked pages that already had potential), not only the refresh itself. A stronger design would match on pre-period volume/position or use a held-out time window and report confidence intervals next to the medians.

**How to make it stronger:** publish the selection rule, pre-period covariates, and n per stratum; avoid headline ratios from tiny buckets.

####**Finding B — “Content health peaks around 61–90 days, then decays if untouched.”**

**Label / outcome:** a health score (or similar composite) by age bucket.

**Methodology question:** Is health a pure observed metric (impressions, CTR, engagement) or partly a product rule? Cross-sectional age curves mix cohort effects (what was published when) with aging. Pages that “survive” to 365+ may be systematically different from pages that died earlier.

**How to make it stronger:** cohort by publish month, show the curve within cohort, and separate “observed traffic metrics” from any internal score so readers know which claims rest on raw measurements.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week-5 already used a client-holdout split. Below we also show a random row split on the same features and metric so the gap is visible. The honest number is the client-holdout one; the random split is the inflated comparison

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv").copy()
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

numeric_feats = [
    "impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "word_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
cat_feats = ["content_type", "main_intent", "position_tier"]

for c in numeric_feats:
    df[c] = pd.to_numeric(df.get(c, 0), errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for c in cat_feats:
    df[c] = df.get(c, "unknown").fillna("unknown").astype(str)

X = pd.concat([
    df[numeric_feats].reset_index(drop=True),
    pd.get_dummies(df[cat_feats], prefix=cat_feats, dtype=float).reset_index(drop=True)
], axis=1)
y = df["is_declining_label"].astype(int)
clients = df["client_id"]

def precision_at_k(y_true, scores, k=50):
    y_true, scores = np.asarray(y_true), np.asarray(scores)
    order = np.argsort(-scores)
    k = min(k, len(order))
    return float(y_true[order[:k]].mean())

def eval_rf(X_tr, y_tr, X_te, y_te):
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    proba = rf.predict_proba(X_te)[:, 1]
    return precision_at_k(y_te, proba, 50), y_te.mean()

# Random row split (less honest)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
p_random, br_random = eval_rf(Xr_tr, yr_tr, Xr_te, yr_te)

# Client-holdout (honest)
u_clients = clients.unique()
tr_c, te_c = train_test_split(u_clients, test_size=0.2, random_state=RANDOM_STATE)
tr_m, te_m = clients.isin(tr_c), clients.isin(te_c)
p_client, br_client = eval_rf(X.loc[tr_m], y.loc[tr_m], X.loc[te_m], y.loc[te_m])

table = pd.DataFrame([
    {"split": "Random rows", "Precision@50": round(p_random, 3), "test_base_rate": round(br_random, 3)},
    {"split": "Client holdout", "Precision@50": round(p_client, 3), "test_base_rate": round(br_client, 3)},
])
display(table)
print("Gap (random - client):", round(p_random - p_client, 3))
print("Honest claim uses the client-holdout number.")


,split,Precision@50,test_base_rate
0,Random rows,0.96,0.542
1,Client holdout,0.68,0.524


Gap (random - client): 0.28
Honest claim uses the client-holdout number.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Final feature set audit against the three leakage types.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_cols = list(X.columns)
print("Feature count:", len(feature_cols))
print("Any column containing 'trend'?", [c for c in feature_cols if "trend" in c.lower()])
print("Any id-like columns?", [c for c in feature_cols if c in ("content_id", "client_id") or c.endswith("_id")])

# Deliberate leak test: add trend_pct and watch Precision@50 jump
df["_trend_pct"] = pd.to_numeric(df.get("trend_pct", 0), errors="coerce").fillna(0)
X_leak = X.copy()
X_leak["LEAK_trend_pct"] = df["_trend_pct"].values

tr_m, te_m = clients.isin(tr_c), clients.isin(te_c)
p_clean, _ = eval_rf(X.loc[tr_m], y.loc[tr_m], X.loc[te_m], y.loc[te_m])
p_leak, _ = eval_rf(X_leak.loc[tr_m], y.loc[tr_m], X_leak.loc[te_m], y.loc[te_m])
print(f"Precision@50 clean: {p_clean:.3f}")
print(f"Precision@50 WITH trend_pct leak: {p_leak:.3f}")
print("Jump confirms the harness detects label leakage.")

Feature count: 27
Any column containing 'trend'? []
Any id-like columns? []
Precision@50 clean: 0.680
Precision@50 WITH trend_pct leak: 1.000
Jump confirms the harness detects label leakage.


*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Label-derived: trend_direction / trend_pct are excluded from features. Adding trend_pct on purpose inflates Precision@50 — that is the confession test.

Future windows: starter data is one snapshot; features and label share the same 90-day view. A production label should sit strictly after the feature window.

Product flags: none in this export; we did not rebuild internal health/priority scores as features.

IDs: used only for the client split, never as model inputs.

## 4. Claim rewrite

###**“Our model predicts which pages will decline and beats the baseline by 3×, so editors should always refresh the top of the queue.”**

###**Honest rewrite:**

“On this anonymized 30k-page snapshot, a random forest ranked declining pages higher than a transparent hand rule under a client-holdout split (Precision@50 improved vs the baseline and vs the base rate). The result is measured and directional on this sample. It is decision-support for prioritising review — not proof that refreshing those pages causes recovery, and not a claim about Google’s ranking algorithm.”

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.